# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [4]:
import os
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from IPython.display import Markdown, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

tavily_search = TavilySearchResults()

# LLM — using DeepSeek via OpenAI-compatible API
llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
    openai_api_base="https://api.deepseek.com/v1"
)

# Safe Tavily wrapper (no tiktoken needed — DeepSeek has 128K context window)
@tool
def tavily_safe_search(query: str) -> str:
    """Search the web for current, up-to-date information using Tavily."""
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily sometimes returns a dict
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    # Trim to ~50K chars (~12K tokens) to leave plenty of room
    max_chars = 50_000
    if len(result_text) > max_chars:
        result_text = result_text[:max_chars]
    return result_text


# Create the LangChain agent
agent = create_agent(model=llm, tools=[tavily_safe_search])


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [5]:
# Your prompt here
prompt_1 = """
Search the web for the latest advancements in generative AI applied to healthcare in 2025. 
Then, produce a **well-structured Markdown summary** with the following sections:

## Generative AI in Healthcare — 2025 Advancements

### 1. Key Breakthroughs
- List 3-5 major breakthroughs with a short description for each.

### 2. Real-World Applications
- Describe how these advancements are being used in hospitals, clinics, or by patients.

### 3. Regulatory & Ethical Developments
- Summarize any notable regulatory approvals (e.g., FDA) or ethical discussions.

### 4. Future Outlook
- Briefly describe what experts expect for the rest of 2025 and beyond.

Make sure every section is populated with concrete, up-to-date information from your web search.
"""

# Run it
result_1 = agent.invoke({"messages": [{"role": "user", "content": prompt_1}]})
response_1 = result_1["messages"][-1].content


In [6]:
display(Markdown(response_1))


I now have comprehensive information to produce the well-structured Markdown summary. Let me compile everything.

# Generative AI in Healthcare — 2025 Advancements

## 1. Key Breakthroughs

### 🧬 **popEVE: Proteome-Wide Rare Disease Diagnosis**
Harvard Medical School and the Centre for Genomic Regulation (CRG) in Barcelona launched **popEVE**, a deep generative AI model published in *Nature Genetics* (November 2025). It scores every possible amino acid substitution across the entire human proteome on a single scale, combining evolutionary data from hundreds of thousands of species with human population genetics. The model identified **123 novel disease-causing mutations** never before linked to developmental disorders — 104 of them observed in just one or two patients. It also addresses the critical issue of ancestry underrepresentation in genetic databases, making rare disease diagnosis faster, cheaper, and more equitable.

### 💊 **MIT's Generative AI-Designed Antibiotics**
Researchers at MIT and McMaster University used generative AI (genetic algorithms and variational autoencoders) to design entirely new antibiotic structures from scratch. From over **36 million candidate molecules**, they synthesized 24 compounds and identified two lead candidates: **NG1**, which eradicates multi-drug-resistant *Neisseria gonorrhoeae* (including strains resistant to first-line therapies), and **DN1**, which targets methicillin-resistant *Staphylococcus aureus* (MRSA). Both cleared infections in mouse models. The compounds are structurally distinct from existing antibiotics, opening new strategies against antimicrobial resistance, which causes nearly 5 million deaths annually. MIT has since received a **$27M ARPA-H grant** to develop 15 new AI-designed antibiotics as preclinical candidates.

### 🏥 **Aidoc's CARE1™ Foundation Model — First FDA-Cleared Clinical Foundation Model**
In February 2025, **Aidoc** received FDA clearance for its **CARE1™ Foundation Model** (Clinical AI Reasoning Engine), marking the **first foundation-model-powered clinical AI** to receive regulatory approval. Unlike traditional single-use AI models, CARE1 is built for scalability, enabling rapid development of new clinical applications. By January 2026, Aidoc secured clearance for a comprehensive AI triage solution covering **11 newly cleared acute indications** (plus 3 existing ones) powered by a single foundation model, achieving a mean sensitivity of 97% and specificity of 98% — with roughly an order-of-magnitude reduction in false alerts compared to single-condition solutions.

### 🎙️ **Ambient AI Clinical Scribes at Scale**
Ambient AI documentation tools became mainstream in 2025. A study published in the *American Journal of Managed Care* found that **62.6% of U.S. hospitals using Epic EHR** had adopted an ambient AI documentation tool by mid-2025 (DAX Copilot, Abridge, and ThinkAndor accounting for over 80% of implementations). Kaiser Permanente Northern California alone used ambient AI scribes across **7,260 physicians for over 2.5 million patient encounters**, saving nearly 16,000 hours of documentation time in one year. A landmark NEJM AI randomized trial across 238 physicians and 14 specialties confirmed significant reductions in documentation time and clinician burnout.

### 🧠 **RecovryAI: First Generative AI Chatbot with FDA Breakthrough Device Designation**
In late 2025, the FDA granted **Breakthrough Device Designation** to **RecovryAI**, an LLM-powered chatbot designed for patients recovering from joint replacement surgery. This marked the **first such designation for a generative AI medical device**, signaling the FDA's willingness to evaluate patient-facing generative AI tools and paving the way for future approvals in this category.

---

## 2. Real-World Applications

### 🏨 **Hospital Adoption & Clinical Documentation**
- **31.5% of U.S. hospitals** were actively using generative AI integrated with EHRs by 2024, with another **24.7% planning adoption within a year** (JAMA Network Open survey of 2,174 hospitals).
- Major teaching hospitals (53.9%) and system-affiliated hospitals (38.5%) led early adoption, while rural and critical access hospitals lagged.
- **68% of physicians** report increased use of AI for clinical documentation, and **72% of AI-using practices** deploy ambient scribes, with two-thirds saving **1–4+ hours per day** on paperwork.

### 🩺 **Clinical Decision Support & Diagnostics**
- **Aidoc's CARE foundation model** is being deployed in emergency departments to triage acute findings across CT imaging, reducing delays in patient flow and easing imaging backlogs.
- **popEVE** is being used in clinical genetics collaborations to identify disease-causing mutations in patients whose variants have never been documented, particularly benefiting clinics with limited resources and patients without parental DNA available.
- **Retrieval-Augmented Generation (RAG)** frameworks are being adopted by health systems to give clinicians AI chatbots that access accurate, organization-specific, up-to-date data — reducing hallucination risks.

### 🧪 **Drug Discovery & Development**
- MIT's generative AI antibiotic pipeline is being expanded under the ARPA-H grant to design 15 new antibiotics as preclinical candidates.
- The pharmaceutical industry is increasingly opening up to AI-driven molecule design, with generative models compressing timelines from insight to intervention.

### 📱 **Patient-Facing Applications**
- **RecovryAI** is being developed as a patient-facing chatbot for post-surgical recovery support.
- AI-powered symptom tracking and supplement recommendations are being deployed in functional and integrative medicine.
- Patient communication tools using generative AI are drafting routine messages, reducing clinician documentation burden.

---

## 3. Regulatory & Ethical Developments

### 🇺🇸 **FDA Landscape**
- The FDA cleared **295 AI/ML-enabled medical devices in 2025**, bringing the cumulative total to **1,451** by end of 2025 (up from just 6 in 2015).
- **Radiology dominates** approvals, accounting for ~76% of all cleared AI devices (1,104 devices), followed by cardiology (~9%).
- **Aidoc's CARE1™** became the first foundation-model-powered clinical AI to receive FDA clearance (February 2025).
- **RecovryAI** received the first FDA **Breakthrough Device Designation** for a generative AI medical device (late 2025).
- The FDA has acknowledged the regulatory challenge: as one STAT News report noted, *"large language models' wide-ranging applications evade simple measures of safety and efficacy,"* challenging traditional device validation approaches. The FDA's tracker page explicitly mentions plans to identify and tag medical devices using generative AI.

### 🇪🇺 **EU AI Act Implementation**
- The **EU AI Act** entered into force with its first provisions (Chapters I and II) on **February 2, 2025**.
- Healthcare AI is classified as **"high-risk"** under the Act, requiring strong oversight, transparency, and continuous monitoring.
- **Unacceptable risk** AI practices (including certain emotion recognition and manipulation tools) were banned from February 2025.
- The Act integrates with existing Medical Device Regulation (MDR) and In Vitro Diagnostic Regulation (IVDR), creating a layered compliance framework.
- Full obligations for high-risk AI systems will apply **36 months after the regulation came into effect** (by 2027).

### ⚖️ **Ethical Considerations**
- Growing concern over **"shadow AI"** — the use of generative AI tools in healthcare outside institutional oversight.
- Calls for **AI literacy training** for clinicians, with medical education increasingly including training on responsible AI use.
- Emerging role of **clinical AI auditors** or medical AI specialists to bridge the gap between healthcare teams and AI systems.
- Hospitals are establishing **AI oversight committees** to evaluate tools before deployment, assessing training data, potential biases, and post-deployment performance.
- Malpractice insurers are beginning to adjust risk assessments based on AI use, potentially offering incentives for validated systems that demonstrably reduce clinical errors.

---

## 4. Future Outlook

### 📈 **From Pilots to Daily Use**
Experts predict 2026 will be the year generative AI moves from pilot projects to **daily clinical use**. As Duke Health's Chief Digital Officer Jeffrey Ferranti put it: *"Generative systems will move from pilot projects to daily use."* The leading ambient AI platform now supports **more than a million weekly clinical encounters** in the U.S. alone — penetration that took EHRs decades to achieve.

### 🤖 **Rise of Agentic AI and Multimodal Models**
- Expect a shift from narrow, single-purpose AI tools to **agentic systems** that orchestrate complex clinical workflows, integrating multimodal data (text, images, genomics, real-time patient vitals).
- Mass General Brigham researchers predict early versions of agentic AI will appear in imaging-heavy specialties like radiology and pathology.
- Generative transformers may forecast diagnoses, treatment response, or disease progression **without needing task-specific labels**.

### 🏗️ **Platform Consolidation**
- The "land of a thousand point solutions" is expected to collapse, with winners being platforms that unify **five or more core capabilities** into a single fabric.
- EHR vendors like Epic are embedding AI directly into their software, including ambient scribes, AI care assistants, and virtual assistants for clinicians, patients, and billing staff.

### 🧭 **From Hype to Reality**
- Mass General Brigham's Hugo Aerts predicts medical AI will move from the *"Peak of Inflated Expectations"* to the early *"Slope of Enlightenment"* on the Gartner Hype Cycle — a healthy reckoning that will separate hype from substance.
- Expect more realism about what AI can and cannot do, with careful measurement of economic impact and clinical outcomes.

### 🌍 **Governance & Compliance Catch-Up**
- Health systems will build out more formal compliance policies to address **shadow AI** risks.
- The EU AI Act's full high-risk obligations will come into force by 2027, driving more rigorous validation and post-market surveillance.
- Expect clearer regulatory and ethical guidelines from bodies like the AMA and FDA, with frameworks addressing disclosure, supervision, and documentation.

### 💡 **The Bottom Line**
2025 was the year generative AI in healthcare moved decisively from experimentation to early adoption at scale. With foundation models receiving FDA clearance, generative AI designing novel therapeutics, and ambient AI becoming standard infrastructure in hospitals, the trajectory is clear: **AI is becoming expected infrastructure — simply part of how healthcare work gets done.** The winners will be those who build workflows around AI from the start, keeping patients and clinicians at the center of innovation.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [7]:
# Your prompt here
prompt_2 = """
Search the web for the top emerging AI startups in 2025 and their innovations. 
Then, produce a **well-structured Markdown summary** with the following sections:

## Top Emerging AI Startups — 2025

### Company Profiles
For each startup, provide:
- **Company Name**
- **Founded** (year)
- **Headquarters** (city, country)
- **Key Product / Innovation** (what makes them unique)
- **Funding** (total raised and latest round, if available)
- **Source** (URL)

Cover at least 5 startups across different AI subfields (e.g., foundation models, AI agents, robotics, biotech, developer tools).

### Trends & Patterns
- Summarize 2-3 common trends you observe among these startups.

Make sure all information is current (2025) and backed by web sources.
"""

# Run it
result_2 = agent.invoke({"messages": [{"role": "user", "content": prompt_2}]})
response_2 = result_2["messages"][-1].content


In [8]:
display(Markdown(response_2))


Now I have enough information to compile a comprehensive markdown summary. Let me put it all together.

# Top Emerging AI Startups — 2025

## Company Profiles

### 1. **Cognition AI** — AI Agents (Software Engineering)

- **Company Name:** Cognition AI (Cognition Labs)
- **Founded:** 2023
- **Headquarters:** San Francisco, California, USA
- **Key Product / Innovation:** **Devin**, an autonomous AI software engineering agent that independently plans, executes, debugs, and deploys complex multi-step coding tasks in a sandboxed environment. Devin works alongside human developers in enterprise teams, handling everything from debugging to deployment. The company also acquired the AI coding tool **Windsurf** in 2025. ARR grew from $1M (Sept 2024) to $73M (June 2025).
- **Funding:** ~$896M total raised. Latest round: $400M Series B at a **$10.2B valuation** (September 2025), led by Founders Fund.
- **Source:** [Contrary Research](https://research.contrary.com/company/cognition) | [Built In SF](https://www.builtinsf.com/articles/cognition-raises-500m-9b-valuation-20250818)

---

### 2. **Sierra** — AI Agents (Enterprise Customer Service)

- **Company Name:** Sierra
- **Founded:** 2023
- **Headquarters:** San Francisco, California, USA
- **Key Product / Innovation:** Builds and implements **AI agents for enterprise customer service**. Sierra's "Agent OS" connects to enterprise systems to autonomously carry out tasks on behalf of customers — from mortgage refinancing and insurance claims to returns and fundraising campaigns. The platform is designed to be less susceptible to hallucinations than typical chatbots. Serves **40% of the Fortune 50** as customers, with billions of customer interactions running on its platform.
- **Funding:** Over $1B total raised. Latest round: $950M Series E at a **$15.8B valuation** (May 2026), led by Tiger Global and GV. Previous round: $350M at $10B (September 2025).
- **Source:** [CNBC](https://www.cnbc.com/2025/09/04/bret-taylor-sierra-ai-startup-salesforce-openai.html) | [Sacra](https://sacra.com/c/sierra)

---

### 3. **FieldAI** — Robotics / Embodied AI

- **Company Name:** FieldAI
- **Founded:** January 2023
- **Headquarters:** Irvine, California, USA
- **Key Product / Innovation:** Builds **general-purpose robotics intelligence** — "risk-aware foundation models" for robots operating in complex real-world industrial environments (construction, energy, mining, urban operations). Unlike approaches that shoehorn LLMs into robotics, FieldAI designs intrinsically risk-aware architectures from the ground up. Led by former NASA JPL and DeepMind veterans, the company has deployed its software across hundreds of real-world industrial environments.
- **Funding:** $405M total raised across Series A and A1 rounds at a **$2B valuation** (August 2025). Investors include Bezos Expeditions, Khosla Ventures, NVIDIA (NVentures), Intel Capital, Temasek, and Gates Frontier.
- **Source:** [GeekWire](https://www.geekwire.com/2025/robotics-startup-fieldai-backed-by-gates-and-bezos-hits-2b-valuation-after-latest-funding) | [FieldAI Press Release](https://www.fieldai.com/news/fieldai-announces-over-400m-in-funds-raised-to-advance-embodied-ai-at-scale)

---

### 4. **Ambience Healthcare** — AI in Healthcare / Biotech

- **Company Name:** Ambience Healthcare
- **Founded:** 2020
- **Headquarters:** San Francisco, California, USA
- **Key Product / Innovation:** An **AI-powered operating system for clinical documentation, coding, and clinical workflows**. Built on proprietary AI reasoning models fine-tuned for healthcare, it generates accurate, compliant, and specialty-aware clinical notes and codes in real time, integrating directly with Epic, Oracle Cerner, and athenahealth EHRs. Supports 100+ medical specialties across outpatient, emergency, and inpatient settings. Used by UCSF Health, Memorial Hermann, and John Muir Health.
- **Funding:** $345M+ total raised. Latest round: $243M Series C at a **$1.25B valuation** (July 2025), co-led by Oak HC/FT and Andreessen Horowitz, with participation from the OpenAI Startup Fund and Kleiner Perkins.
- **Source:** [SiliconANGLE](https://siliconangle.com/2025/07/29/ai-native-clinical-documentation-startup-ambience-healthcare-raises-243m) | [Ambience Healthcare](https://www.ambiencehealthcare.com/blog/ambience-healthcare-announces-243-million-series-c-to-scale-its-ai-platform-for-health-systems)

---

### 5. **Anysphere (Cursor)** — Developer Tools / AI Coding

- **Company Name:** Anysphere (maker of Cursor)
- **Founded:** December 2022
- **Headquarters:** San Francisco, California, USA
- **Key Product / Innovation:** **Cursor**, an AI-first code editor built from the ground up as an AI-native IDE. It supports natural language commands, smart autocompletion, chat-based assistance, and multi-file edits powered by custom-trained models. Cursor reached $100M ARR in its first 12 months (fastest SaaS company ever), surpassed $1B ARR within 21 months, and now serves 1M+ paying customers including 70% of the Fortune 1000.
- **Funding:** $3.3B+ total raised. Latest round: $2.3B Series D at a **$29.3B valuation** (November 2025), co-led by Accel and Coatue, with participation from NVIDIA and Google.
- **Source:** [Forge Global](https://forgeglobal.com/anysphere_stock) | [The Next Web](https://thenextweb.com/news/cursor-anysphere-2-billion-funding-50-billion-valuation-ai-coding)

---

### 6. **Thinking Machines Lab** — Foundation Models

- **Company Name:** Thinking Machines Lab
- **Founded:** February 2025
- **Headquarters:** San Francisco, California, USA
- **Key Product / Innovation:** Founded by **Mira Murati** (former OpenAI CTO), this startup is building toward **"collaborative general intelligence"** — AI systems that work as intelligent teammates rather than autonomous agents. The company focuses on multimodal, customizable, user-driven AI. Its first product, **Tinker** (launched October 2025), lets developers fine-tune AI models for specific tasks without the cost of traditional distributed training. Nearly two-thirds of the founding team came from OpenAI.
- **Funding:** $2B seed round at a **$12B valuation** (July 2025) — one of the largest seed rounds in Silicon Valley history, led by Andreessen Horowitz with participation from NVIDIA, AMD, Cisco, and Jane Street. Reportedly seeking $5B more at a $50B valuation.
- **Source:** [Wikipedia](https://en.wikipedia.org/wiki/Thinking_Machines_Lab) | [Built In](https://builtin.com/articles/what-is-thinking-machines-lab)

---

### 7. **Decart** — AI Infrastructure / Real-Time World Models

- **Company Name:** Decart
- **Founded:** 2023
- **Headquarters:** San Francisco, California, USA (founded in Israel)
- **Key Product / Innovation:** An **efficiency-first AI research lab** building real-time video and world models, plus ultra-optimized inference infrastructure. Its **DOS 2.0** platform delivers 1,600+ tokens per second for agentic inference and 100+ FPS for world models. Products include **Lucy** (immersive real-time experiences) and **Oasis** (physical AI). Amazon is a strategic customer; NVIDIA is both investor and business partner.
- **Funding:** $450M+ total raised. Latest round: $300M Series B at a **$4B valuation** (May 2026), led by Radical Ventures with NVIDIA, Sequoia, Benchmark, Adobe, and Toyota Ventures.
- **Source:** [Calcalist](https://www.calcalistech.com/ctechnews/article/sjt9ncukgl) | [The Next Web](https://thenextweb.com/news/decart-300-million-radical-ventures-world-models)

---

### 8. **Cerebras Systems** — AI Hardware / Chips

- **Company Name:** Cerebras Systems
- **Founded:** 2015
- **Headquarters:** Sunnyvale, California, USA
- **Key Product / Innovation:** Builds the **world's largest AI chips** — wafer-scale processors (WSE-3) containing 4 trillion transistors, 19x more than NVIDIA's Blackwell B200. The large on-chip memory pool eliminates the need for off-chip HBM memory, dramatically speeding up AI inference and training. Cerebras is the #1 inference provider on Hugging Face with 5M+ monthly requests, and serves AWS, Meta, IBM, Mistral, Cognition, and the US Department of Defense.
- **Funding:** $2.8B+ total raised. Latest round: $1.1B Series G at an **$8.1B valuation** (September 2025), led by Fidelity and Atreides Management. Later raised an additional $1B at a $23B valuation (February 2026).
- **Source:** [Cerebras Press Release](https://www.cerebras.ai/press-release/series-g) | [SiliconANGLE](https://siliconangle.com/2026/02/04/cerebras-raises-another-1b-wafer-size-ai-chip)

---

## Trends & Patterns

### 1. **Massive Capital Concentration in AI**
In 2025, AI startups raised nearly **$150 billion**, accounting for more than **40% of global venture capital** — the first time AI has secured a majority share of VC investment. Foundation model companies alone raised $80 billion. The 10 largest AI deals represented 41.2% of all VC activity, a level of concentration never seen before in venture capital history. This reflects an unprecedented "winner-take-most" dynamic where mega-rounds at $10B+ valuations have become routine for top-tier AI startups.

### 2. **Shift from Chatbots to Autonomous Agents**
Across nearly every subfield, the dominant trend is the move from conversational AI to **autonomous, agentic systems** that execute multi-step tasks independently. Cognition's Devin autonomously writes and deploys code, Sierra's agents handle end-to-end customer service transactions, FieldAI's robots operate in hazardous industrial environments without human intervention, and Thinking Machines Lab is explicitly building toward "collaborative general intelligence" rather than simple chat interfaces. The market is clearly betting that agents — not chatbots — are where the real value lies.

### 3. **Vertical AI Specialization and Infrastructure Build-Out**
While foundation model companies capture headlines, the fastest-growing startups are those applying AI to **specific verticals** (healthcare documentation, property management, code review, industrial robotics) or building the **underlying infrastructure** (wafer-scale chips, optimized inference platforms, real-time world models). Companies like Ambience Healthcare, EliseAI, and Greptile are proving that deeply specialized AI solutions for narrow domains can achieve unicorn status faster than general-purpose AI tools. Simultaneously, infrastructure players like Cerebras and Decart are racing to solve the compute bottleneck that constrains all AI development.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [9]:
# Your prompt here
prompt_3 = """
Search the web and compare **OpenAI's ChatGPT Team** and **Anthropic's Claude Pro**. 
Produce a **well-structured Markdown summary** with the following sections:

## ChatGPT Team vs. Claude Pro — 2025 Comparison

### 1. Overview
- Brief description of each product (1-2 sentences each).

### 2. Key Features
| Feature | ChatGPT Team | Claude Pro |
|---------|-------------|------------|
| Model(s) used | ... | ... |
| Context window | ... | ... |
| Multimodal support | ... | ... |
| Collaboration features | ... | ... |
| Integrations | ... | ... |

### 3. Pricing
- List the current pricing for each product (monthly, annual, team plans).

### 4. Reviews & Reception
- Summarize what users and reviewers are saying about each product.

### 5. Verdict
- Provide a balanced recommendation: which product suits which type of user?

Make sure all information is current as of 2025.
"""

# Run it
result_3 = agent.invoke({"messages": [{"role": "user", "content": prompt_3}]})
response_3 = result_3["messages"][-1].content


In [10]:
display(Markdown(response_3))


Now I have comprehensive information to compile the comparison. Let me put it all together.

# ChatGPT Team vs. Claude Pro — 2025 Comparison

## 1. Overview

**ChatGPT Team** (renamed **ChatGPT Business** in August 2025) is OpenAI's collaborative workspace plan designed for small-to-medium teams and startups. It bundles all ChatGPT Plus features with shared workspaces, admin controls, higher usage limits, and integrations with business tools like Slack, Google Drive, and SharePoint — making it a secure, managed AI environment for organizations of 2+ users.

**Claude Pro** is Anthropic's individual premium subscription for power users. It provides significantly higher usage limits than the free tier, access to Anthropic's most advanced models (Opus and Sonnet), Claude Code (terminal-based coding agent), Projects, Artifacts, and priority access during peak traffic — all focused on deep reasoning, long-form writing, and sophisticated coding.

---

## 2. Key Features

| Feature | ChatGPT Team | Claude Pro |
|---------|-------------|------------|
| **Model(s) used** | GPT-5 (default), GPT-5 Thinking, GPT-5 Thinking Pro, plus access to o3/o4-mini and legacy models | Claude Opus 4.x, Claude Sonnet 4.x, Claude Haiku 4.x (full model access) |
| **Context window** | ~128K tokens (GPT-5 Fast), ~196K tokens (GPT-5 Thinking); 32K for legacy models | 200K tokens standard (1M beta available via API) |
| **Multimodal support** | ✅ Full multimodal: image generation (DALL-E/GPT-image), voice mode, image input, video (Sora), audio transcription | ⚠️ Partial: image *analysis* and vision input supported, but **no native image generation** or voice mode |
| **Collaboration features** | Shared workspaces, project sharing (Can Chat / Can Edit permissions), shared custom GPTs, admin console, unified billing, team folders | Projects (shared), Artifacts (shared), but primarily designed for individual use; team collaboration requires Claude Team plan |
| **Integrations** | Slack, Google Drive, SharePoint, HubSpot, Canva, Notion, GitHub, Asana, MCP custom connectors, Google Workspace (Gmail, Calendar, Contacts) | Google Workspace (Gmail, Calendar, Docs), Microsoft 365 (SharePoint, OneDrive, Outlook), Slack, GitHub, MCP connectors, Claude for Chrome, Claude for Microsoft 365 |

---

## 3. Pricing

### ChatGPT Team
- **Monthly billing:** $30 per user/month (minimum 2 users)
- **Annual billing:** $25 per user/month (minimum 2 users)
- **Note:** Renamed to **ChatGPT Business** in August 2025. In April 2026, pricing was reduced to $20/user/month (annual) or $25/user/month (monthly).
- Includes all Plus features plus team management, admin controls, and business integrations.

### Claude Pro
- **Monthly billing:** $20/month
- **Annual billing:** $17/month (billed as $200/year)
- **Note:** Claude Pro is an individual plan — no team features included. For teams, Anthropic offers **Claude Team** at $25–$30/user/month (standard seats) or $100–$125/user/month (premium seats with Claude Code).

---

## 4. Reviews & Reception

### ChatGPT Team
- **Positive reception:** Businesses praise the flat per-seat pricing, data privacy guarantees (no training on business data), and the breadth of features — from image generation to deep research to voice mode. The admin console, shared custom GPTs, and project sharing have been well-received for team collaboration. OpenAI reports ~5 million paying business seats as of mid-2025, with 92% of Fortune 500 companies having active deployments.
- **Criticisms:** Some users note the context window is smaller than competitors (32K for legacy models, ~128K for GPT-5), which can be limiting for long-document work. The plan lacks enterprise-grade features like SSO, SCIM provisioning, and audit logs (those require Enterprise). Some reviewers find the interface cluttered compared to Claude's cleaner design.

### Claude Pro
- **Positive reception:** Users consistently praise Claude's natural, human-like writing quality, exceptional coding capabilities (Claude Code is widely considered best-in-class), and massive 200K context window. G2 rates Claude 4.4/5 overall, with 93% satisfaction on natural conversation and 92% on ease of use. Developers and writers frequently cite Claude as superior for long-form content, complex reasoning, and code architecture.
- **Criticisms:** The lack of native image generation and voice mode is a notable gap. Usage limits (approximately 45 messages per 5-hour window) can feel restrictive for heavy users. Some users note Claude can be overly cautious or "safe" in its responses. No built-in web browsing in the standard interface (though web search is available in certain configurations).

---

## 5. Verdict

**Choose ChatGPT Team if:**
- You need a **collaborative team workspace** with admin controls, shared projects, and unified billing
- Your work requires **multimodal capabilities** — image generation, voice mode, video (Sora), or audio transcription
- You want **broad integrations** with Slack, Google Drive, SharePoint, HubSpot, and other business tools
- You need a **general-purpose AI assistant** that handles everything from content creation to research to quick coding tasks
- Your team values **breadth of features** over depth in any single area

**Choose Claude Pro if:**
- You are an **individual power user** (writer, developer, researcher, analyst) who doesn't need team management features
- Your work involves **long documents, complex reasoning, or sophisticated coding** — Claude's 200K context window and Claude Code are unmatched
- You prioritize **writing quality and natural language** over image generation or voice features
- You want a **cleaner, more focused interface** without the feature sprawl of ChatGPT
- You need **deep analytical thinking** and careful, nuanced responses over speed and breadth

**Bottom line:** These are complementary tools rather than direct competitors. ChatGPT Team is the better choice for organizations that need a managed, collaborative AI workspace with broad multimodal capabilities. Claude Pro is the better choice for individual professionals who need deep reasoning, exceptional writing, and powerful coding assistance. Many power users subscribe to both — using ChatGPT for image generation, voice, and broad tasks, and Claude for the "hard stuff" — coding, long-form analysis, and nuanced writing.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
